In [58]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder.appName("FinalProject").getOrCreate()

# 1. Read training data

In [59]:
#Instruction: You should read this data into a standard pandas data frame using the pd.read_csv() function.
df_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [60]:
#Instruction: Convert this to a spark data frame
spark_df = spark.createDataFrame(df_pd)
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [61]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

# 2. Pre-processing

Instruction:
We are going to treat the Power_Zone_3 variable as our response variable.
We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading
is going to go offline in the future and we need to be able to predict that value appropriately.)

In [62]:
#cast hour and rename response variable
sql_transformer = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double, Power_Zone_3 AS label
    FROM __THIS__""")
#예측변수인 Power_Zone_3를 label이라는 이름으로 복사

Instruction: The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType.
Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

In [63]:
#create new binary variable
hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_double", outputCol = "Hour_binary")

Instruction: One-hot encode the Month column

In [64]:
month_encoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_encoded"])

Instruction: Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and
Diffuse_Flows columns.

In [65]:
#Instruction: Use a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol = "pca_input")

In [66]:
#Instruction: We’ll use two PCs in our transformation.
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features") #define PCA transformer

Instruction: Use VectorAssembler() to put your predictors into a features. Use the
* two fitted PCA features
* binary Hour variable
* Power_Zone_1
* Power_Zone_2
* Month indicator variables

In [67]:
feature_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_binary", "Power_Zone_1", "Power_Zone_2", "Month_encoded"],
    outputCol = "features")

Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.

In [68]:
lr = LinearRegression(featuresCol = "features", labelCol = "label", predictionCol = "prediction")

Instruction: The transformations below should each use an MLlib function that can be put into a pipeline

In [69]:
#Build full pipeline
pipeline = Pipeline(stages = [sql_transformer, hour_binarizer, month_encoder, pca_assembler, pca, feature_assembler, lr])

# 3. Hyperparameter Tuning

Instruction: Now you’ll then use the CrossValidator() function and the LinearRegression() function to fit an
elastic net model.

In [70]:
#define evaluator of model performance
evaluator = RegressionEvaluator(labelCol = "label", predictionCol = "prediction", metricName = "rmse")

Instruction: You should do the following grid for the regParam and elasticNetParam: All combinations of
* regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
* elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

In [72]:
#define parameter grid
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = (ParamGridBuilder().addGrid(lr.regParam, param_values).addGrid(lr.elasticNetParam, param_values).build())

Instruction: Now fit the model using 5-fold CV with rmse as your criterion!

In [73]:
#set up 5-fold cross validation
cv = CrossValidator(estimator = pipeline,
                    estimatorParamMaps = param_grid,
                    evaluator = evaluator,
                    numFolds = 5,
                    seed = 123)

In [77]:
#fit the model
cv_model = cv.fit(spark_df)

26/04/28 16:25:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/28 16:25:28 WARN Instrumentation: [49ae09e7] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 16:25:29 WARN Instrumentation: [49ae09e7] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 16:25:31 WARN Instrumentation: [f35dae81] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 16:25:31 WARN Instrumentation: [f35dae81] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/28 16:25:33 WARN Instrumentation: [66ad31f2] regParam is zero, which might cause numerical instability and overfitting.
26/04/28 16:25:33 WARN Instrumentation: [66ad31f2] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Instruction: Report the optimal values chosen for the tuning parameters

In [78]:
#report optimal tuning parameters
best_model = cv_model.bestModel
best_lr_model = best_model.stages[-1]

best_reg_param = best_lr_model.getRegParam()
best_elastic_net_param = best_lr_model.getElasticNetParam()

print("Best regParam:", best_reg_param)
print("Best elasticNetParam:", best_elastic_net_param)

Best regParam: 0.05
Best elasticNetParam: 0.1


Instruction: Report the CV error

In [79]:
#cv_model.avgMetrics에는 각 parameter 조합별 평균 CV RMSE가 저장된다. 가장 작은 값이 최적 CV error이다.
cv_errors = cv_model.avgMetrics
best_cv_error = min(cv_errors)

print("Best CV RMSE:", best_cv_error)

Best CV RMSE: 2147.8113505767933


Instruction: Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and
evaluating on the entire training set

In [80]:
#최종 선택된 모델을 training data에 적용
training_predictions = cv_model.transform(spark_df)
training_rmse = evaluator.evaluate(training_predictions)

print("Training RMSE:", training_rmse)

Training RMSE: 2147.0973169293934


Instruction: Take the outputted transformations from the model (the predictions) and create a residual column
(label - prediction). The .withColumn() method is handy here. Print out a data frame with these
residuals, the label column, and the predictions

In [82]:
#예측 결과에서 잔차 칼럼을 만듦
training_predictions_with_residuals = training_predictions.withColumn("residual", F.col("label") - F.col("prediction"))

training_predictions_with_residuals.select("label", "prediction", "residual").show(20)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20878.85066078923|-637.8868007892306|
|20131.08434|18660.227266543785|1470.8570734562163|
|19668.43373| 18204.75215311428|1463.6815768857196|
|18899.27711|17590.648498338862|1308.6286116611373|
|18442.40964|16997.301986456572|1445.1076535434295|
|18130.12048|16517.686723493975|1612.4337565060268|
|17945.06024|16093.246141053161|1851.8140989468375|
|17459.27711|15722.695360253583|1736.5817497464159|
|17025.54217|15271.043828662481|1754.4983413375194|
|16794.21687| 14938.34876466529|  1855.86810533471|
|16638.07229|14652.383721423503|1985.6885685764973|
|16395.18072|14414.900662730051|1980.2800572699489|
|16117.59036|14082.889275686459|2034.7010843135413|
| 15822.6506| 13624.88209143571|2197.7685085642916|
|15672.28916|13450.340775467957| 2221.948384532043|
|15597.10843|13302.246394561593|2294.8620354384075|
|15510.36145

# Streaming Part